# Constructing Summary Matrix
Deviance seems to be a measure used to select highly variable genes. We'll try to use this same measure to select highly variable coordinates. However, to this end, we need an object that describes all reads for all coordinates across all samples. In this notebook, we'll construct a function to this end and test it.

## Imports

In [2]:
### Enabling autoreload ##
%load_ext autoreload
%autoreload 2

In [3]:
### Directories and Files ###
root = '../../../'
data_dir = '/Users/oliviersmeets/Desktop/University/Master Internship 1/Data Handling/Data/ramani_cool_files' # Importing was not working with relative root+'..' approach, so replace with local data
save_dir = root+'Generated Data/Single-to-single/sc_scc_ramani'

In [ ]:
### Imports ###
import sys
import numpy as np
import pandas as pd
import cooler
sys.path.append(root+'Scripts')
from hicdatautils import import_cool_dir, fetch_region, get_cool_name

In [5]:
### Importing coolers ###
clrs = import_cool_dir(data_dir)

## Main Function

In [ ]:
def compile_hic_reads(coolers: list[cooler.Cooler], chrom: str, ) -> pd.DataFrame:
    '''
        Takes a list of coolers and a chromosome and returns a dataframe
        summarizing the data in all coolers, with genomic coordinates as
        the indices as (i, j) and read counts per cell in the columns

        Parameters
        ----------
        coolers : list[cooler.Cooler]
            List of coolers to consider
        chrom : str
            Chromosome to consider.
        
        Returns
        -------
        compiled_reads : pd.DataFrame
            Summary of reads with coordinates as indices and reads per
            cell in columns
    '''
    # Set reference binsize and coordinates
    ref_clr = coolers[0]
    ref_binsize = ref_clr.binsize
    ref_region = fetch_region(ref_clr, chrom=chrom)

    maskrows, maskcols = np.triu_indices(ref_region.shape[0])
    
        # Create multi-index
    index = pd.MultiIndex.from_arrays(
        [maskrows, maskcols],
        names=["bin_i", "bin_j"]
    )

    # Dict for converting later
    compiled_dict = {}

    # Loop through each cooler
    for clr in coolers:
        # Check binsize and get name
        if clr.binsize != ref_binsize:
            raise ValueError("Binsize not homogenous across input coolers.")
        name = get_cool_name(clr)

        # Fetch region
        region = fetch_region(clr, chrom=chrom)

        # Extract points and coordinates
        values = region[maskrows, maskcols]

        # Add values to dict
        compiled_dict[name] = values
    
    # Convert to dataframe
    compiled_reads = pd.DataFrame(compiled_dict, index=index)

    return compiled_reads

## Testing

In [20]:
test = [
    [1, 2, 3, 4, 5],
    [2, 1, 2, 3, 4],
    [3, 2, 1, 2, 3],
    [4, 3, 2, 1, 2],
    [5, 4, 3, 2, 1]
]

mask = np.triu(test)
mask

array([[1, 2, 3, 4, 5],
       [0, 1, 2, 3, 4],
       [0, 0, 1, 2, 3],
       [0, 0, 0, 1, 2],
       [0, 0, 0, 0, 1]])

In [30]:
compile_hic_reads(clrs, "chr1")

cell231_HAP1_  cell447_HAP1_  cell470_HAP1_  cell312_HAP1_  \
bin_i bin_j                                                               
0     0           1.028661       0.285375       0.312262       0.341930   
      1           0.753102       0.239738       0.315509       0.320945   
      2           0.726784       0.108721       0.119932       0.155017   
      3           0.632567       0.007453       0.003274       0.029000   
      4           0.442993       0.000398       0.000182       0.000479   
...                    ...            ...            ...            ...   
247   248         0.981592       0.637260       0.776159       0.720903   
      249         0.168058       0.020714       0.055638       0.028208   
248   248         1.921122       1.106228       1.271637       1.473078   
      249         0.173072       0.092492       0.091935       0.090360   
249   249         0.026762       0.007243       0.011108       0.011954   

             cell29_HAP1_  cell264_HAP1_  cell181_HeLa_  cell412_HAP1_  \
bin_i bin_j                                                              
0     0          0.861736       0.736400       0.670246       0.608400   
      1          0.668721       0.622554       0.562324       0.406955   
      2          0.619934       0.388824       0.472816       0.271594   
      3          0.553550       0.133087       0.359217       0.078112   
      4          0.289326       0.107370       0.218549       0.007543   
...                   ...            ...            ...            ...   
247   248        0.913622       0.885066       0.803893       0.875235   
      249        0.152153       0.157705       0.149175       0.159892   
248   248        1.800165       1.698459       1.553649       1.617354   
      249        0.179330       0.164042       0.156364       0.161262   
249   249        0.032087       0.025972       0.026075       0.016697   

             cell303_HeLa_  cell149_HeLa_  ...  cell112_HeLa_  cell36_HeLa_  \
bin_i bin_j                                ...                                
0     0           0.842762       0.721870  ...       0.643486      0.835233   
      1           0.640093       0.546353  ...       0.548589      0.435758   
      2           0.693362       0.264746  ...       0.378279      0.108883   
      3           0.608592       0.160327  ...       0.307503      0.066060   
      4           0.366758       0.036000  ...       0.115026      0.015304   
...                    ...            ...  ...            ...           ...   
247   248         0.909312       0.809091  ...       0.766412      0.704328   
      249         0.181240       0.160814  ...       0.143059      0.173037   
248   248         1.836289       1.593841  ...       1.503343      1.502313   
      249         0.198141       0.166016  ...       0.153389      0.178701   
249   249         0.035450       0.033401  ...       0.026703      0.029819   

             cell147_HeLa_  cell458_HeLa_  cell381_HAP1_  cell63_HeLa_  \
bin_i bin_j                                                              
0     0           0.709428       0.348329       0.685499      0.813366   
      1           0.618643       0.634908       0.424210      0.678678   
      2           0.508819       0.644894       0.107575      0.455684   
      3           0.291398       0.302878       0.006719      0.166363   
      4           0.090198       0.004904       0.002001      0.109213   
...                    ...            ...            ...           ...   
247   248         0.823853       0.829071       0.802316      0.909268   
      249         0.144177       0.147146       0.064241      0.181027   
248   248         1.590686       1.144733       1.619781      1.731761   
      249         0.157237       0.153270       0.103699      0.178153   
249   249         0.027928       0.008450       0.024464      0.034043   

             cell54_HeLa_  cell549_K562_  cell219_HeLa_  cell295_HAP1_  
bin_i bin_

This function works perfectly, which will allow us to test our deviance approach.

In [47]:
np.load(f'/Users/oliviersmeets/Downloads/Ramani/output/embed/exp_zinb3_0_origin.npy')[1]

array([-28.6831    ,  12.562384  ,  11.843275  , -47.454533  ,
        12.171972  ,   9.055595  ,  16.228642  ,  12.277258  ,
         9.0446    ,   8.903729  ,   8.164591  ,  12.561667  ,
         4.6373634 ,  21.182625  ,   4.6795497 ,  20.298597  ,
         5.3502264 ,  10.241135  ,  16.040754  ,   4.323254  ,
        10.8725    ,   0.56852853,   4.633085  ,  18.811453  ,
        17.604517  ,   8.984237  ,  10.063358  ,  10.663313  ,
         7.7256346 ,   4.4363394 ,   3.6992588 ,   4.3326464 ,
         7.193202  ,   1.8858899 ,   7.4612646 ,  19.53689   ,
         6.17878   ,   3.3824348 ,  10.117825  ,   4.9667625 ,
        17.031868  ,  13.880297  ,   1.593741  ,  12.101733  ,
        15.5392885 ,  35.098457  ,  12.210509  ,   2.7820349 ,
        10.962417  ,  13.672264  ,  13.287184  ,  17.8582    ,
         8.679557  ,  15.349984  ,  14.929257  ,   9.289721  ,
         9.982518  ,  -0.54826343,  -1.6867552 ,   5.276679  ,
        12.21797   ,  14.70562   ,  13.848177  ,   1.79